In [8]:
import torch

pt_path = "outputs/attention_dumps/bitcoin_synthetic_additive_toy_forward_attn.pt"
# PyTorch >=2.6 defaults to weights_only=True; this file is a mixed payload.
data = torch.load(pt_path, map_location="cpu", weights_only=False)

print("Top-level keys:", list(data.keys()))
print("Metrics:", data.get("metrics", {}))

# Choose which split to inspect: "tune" or "eval"
split = "tune"
records = data["attention"][split]
print(f"\nLoaded {len(records)} datapoints from attention['{split}']")

# Print per-datapoint summary and a small head-wise preview
for rec in records:
    step = rec["step_index"]
    ctx = rec["context_size"]
    eval_pos = rec["eval_pos"]
    attn = rec["final_attention_per_head"]  # [num_heads, n_total_rows, n_context_rows]

    print(f"\nDatapoint {step}: context_size={ctx}, eval_pos={eval_pos}, attn_shape={tuple(attn.shape)}")

    # Last query row attention (the current prediction row) by head
    # shape -> [num_heads, n_context_rows]
    last_query = attn[:, -1, :]
    print("  last_query_by_head shape:", tuple(last_query.shape))

    # Print a compact preview: first 3 heads x first 8 context positions
    h_show = min(3, last_query.shape[0])
    c_show = min(8, last_query.shape[1])
    preview = last_query[:h_show, :c_show]
    print(f"  preview first {h_show} heads x first {c_show} context cols:")
    print(preview)

    # Optional: uncomment to stop early
    # if step >= 4:
    #     break

Top-level keys: ['config_path', 'dataset_key', 'use_text', 'splits', 'metrics', 'predictions', 'attention']
Metrics: {'tune_rmse': 572.2463957946786, 'tune_mae': 366.65509033203125, 'eval_rmse': 431.13308197701554, 'eval_mae': 284.697265625}

Loaded 10 datapoints from attention['tune']

Datapoint 0: context_size=37, eval_pos=37, attn_shape=(8, 38, 37)
  last_query_by_head shape: (8, 37)
  preview first 3 heads x first 8 context cols:
tensor([[0.0296, 0.0289, 0.0274, 0.1236, 0.0469, 0.0123, 0.0117, 0.0029],
        [0.0870, 0.1260, 0.0927, 0.0504, 0.0123, 0.0064, 0.0058, 0.0033],
        [0.0580, 0.0778, 0.0759, 0.0174, 0.0021, 0.0005, 0.0004, 0.0002]])

Datapoint 1: context_size=38, eval_pos=38, attn_shape=(8, 39, 38)
  last_query_by_head shape: (8, 38)
  preview first 3 heads x first 8 context cols:
tensor([[0.0254, 0.0281, 0.0241, 0.0920, 0.0401, 0.0118, 0.0129, 0.0038],
        [0.0869, 0.1203, 0.0932, 0.0652, 0.0137, 0.0074, 0.0070, 0.0041],
        [0.0494, 0.0646, 0.0606, 0.0268,

In [9]:
import torch


def compare_attention_vectors_line_by_line(
    records,
    datapoint_x: int,
    datapoint_y: int,
    *,
    query_row: int = -1,
    max_context_cols: int | None = 20,
    atol: float = 1e-6,
):
    """
    Compare attention vectors for two datapoints, head by head, line by line.

    Args:
        records: data['attention']['tune'] or data['attention']['eval']
        datapoint_x: step_index for first datapoint
        datapoint_y: step_index for second datapoint
        query_row: which row in [n_total_rows] to compare (default -1 = current query row)
        max_context_cols: print only first N context positions (None = print all)
        atol: tolerance for equality checks
    """

    idx_map = {int(r['step_index']): r for r in records}
    if datapoint_x not in idx_map:
        raise ValueError(f"datapoint_x={datapoint_x} not found")
    if datapoint_y not in idx_map:
        raise ValueError(f"datapoint_y={datapoint_y} not found")

    rec_x = idx_map[datapoint_x]
    rec_y = idx_map[datapoint_y]

    attn_x = rec_x['final_attention_per_head']
    attn_y = rec_y['final_attention_per_head']
    if not isinstance(attn_x, torch.Tensor):
        attn_x = torch.as_tensor(attn_x)
    if not isinstance(attn_y, torch.Tensor):
        attn_y = torch.as_tensor(attn_y)

    # Shapes: [num_heads, n_total_rows, n_context_rows]
    if attn_x.shape[0] != attn_y.shape[0]:
        raise ValueError(f"Different number of heads: {attn_x.shape[0]} vs {attn_y.shape[0]}")

    qx = query_row if query_row >= 0 else attn_x.shape[1] + query_row
    qy = query_row if query_row >= 0 else attn_y.shape[1] + query_row
    if qx < 0 or qx >= attn_x.shape[1]:
        raise ValueError(f"query_row out of range for datapoint_x: {query_row}")
    if qy < 0 or qy >= attn_y.shape[1]:
        raise ValueError(f"query_row out of range for datapoint_y: {query_row}")

    vec_x = attn_x[:, qx, :].float()  # [H, Cx]
    vec_y = attn_y[:, qy, :].float()  # [H, Cy]

    # Compare shared prefix if context sizes differ.
    common_c = min(vec_x.shape[1], vec_y.shape[1])
    vec_x = vec_x[:, :common_c]
    vec_y = vec_y[:, :common_c]

    if max_context_cols is not None:
        common_c = min(common_c, max_context_cols)
        vec_x = vec_x[:, :common_c]
        vec_y = vec_y[:, :common_c]

    print(
        f"Compare datapoint {datapoint_x} vs {datapoint_y} | "
        f"heads={vec_x.shape[0]} | query_row={query_row} | compared_context={vec_x.shape[1]}"
    )

    for h in range(vec_x.shape[0]):
        hx = vec_x[h]
        hy = vec_y[h]
        diff = hx - hy
        l1 = diff.abs().sum().item()
        l2 = torch.norm(diff, p=2).item()
        max_abs = diff.abs().max().item()
        equal = torch.allclose(hx, hy, atol=atol, rtol=0.0)

        print(f"\nHead {h}")
        print("  x:", " ".join(f"{v:.6f}" for v in hx.tolist()))
        print("  y:", " ".join(f"{v:.6f}" for v in hy.tolist()))
        print("  d:", " ".join(f"{v:+.6f}" for v in diff.tolist()))
        print(f"  stats -> allclose={equal}, L1={l1:.6f}, L2={l2:.6f}, max|d|={max_abs:.6f}")


# Example:
# records = data['attention']['tune']
# compare_attention_vectors_line_by_line(records, datapoint_x=3, datapoint_y=8, query_row=-1, max_context_cols=12)


In [13]:

compare_two_attention_pt_same_head_full_vector(
    "outputs/attention_dumps/bitcoin_synthetic_additive_toy_forward_attn.pt",
    "outputs/attention_dumps/bitcoin_synthetic_additive_forward_attn.pt",
    split="tune",      # or "eval"
    datapoint_idx=1,  # step_index
    head_idx=1,
    query_row=-1,      # current query row
)

Split=tune | datapoint=1 | head=1 | query_row=-1
Vector length=38 | allclose=False
Stats: L1=0.385881, L2=0.117450, max|d|=0.087456, mean|d|=0.010155

A:
0.086900 0.120310 0.093216 0.065216 0.013710 0.007409 0.006995 0.004147 0.006237 0.003604 0.004832 0.007040 0.004308 0.004308 0.008310 0.004420 0.006197 0.005596 0.005184 0.005704 0.008205 0.016300 0.030334 0.020475 0.024265 0.010522 0.006995 0.025863 0.030045 0.032852 0.027001 0.037560 0.038043 0.090868 0.031618 0.032956 0.031517 0.040938

B:
0.044160 0.106795 0.101489 0.152673 0.030003 0.007234 0.004984 0.003082 0.003554 0.002637 0.002700 0.005281 0.001834 0.002073 0.008124 0.002037 0.003345 0.004325 0.004235 0.007916 0.007435 0.012069 0.016503 0.016753 0.037201 0.036377 0.044590 0.014203 0.021152 0.031470 0.029322 0.031259 0.036234 0.066330 0.024335 0.030097 0.019592 0.026600

A-B:
+0.042739 +0.013515 -0.008272 -0.087456 -0.016293 +0.000175 +0.002011 +0.001065 +0.002683 +0.000967 +0.002133 +0.001759 +0.002475 +0.002236 +0.000186 +0

{'vec_a': tensor([0.0869, 0.1203, 0.0932, 0.0652, 0.0137, 0.0074, 0.0070, 0.0041, 0.0062,
         0.0036, 0.0048, 0.0070, 0.0043, 0.0043, 0.0083, 0.0044, 0.0062, 0.0056,
         0.0052, 0.0057, 0.0082, 0.0163, 0.0303, 0.0205, 0.0243, 0.0105, 0.0070,
         0.0259, 0.0300, 0.0329, 0.0270, 0.0376, 0.0380, 0.0909, 0.0316, 0.0330,
         0.0315, 0.0409]),
 'vec_b': tensor([0.0442, 0.1068, 0.1015, 0.1527, 0.0300, 0.0072, 0.0050, 0.0031, 0.0036,
         0.0026, 0.0027, 0.0053, 0.0018, 0.0021, 0.0081, 0.0020, 0.0033, 0.0043,
         0.0042, 0.0079, 0.0074, 0.0121, 0.0165, 0.0168, 0.0372, 0.0364, 0.0446,
         0.0142, 0.0212, 0.0315, 0.0293, 0.0313, 0.0362, 0.0663, 0.0243, 0.0301,
         0.0196, 0.0266]),
 'diff': tensor([ 0.0427,  0.0135, -0.0083, -0.0875, -0.0163,  0.0002,  0.0020,  0.0011,
          0.0027,  0.0010,  0.0021,  0.0018,  0.0025,  0.0022,  0.0002,  0.0024,
          0.0029,  0.0013,  0.0009, -0.0022,  0.0008,  0.0042,  0.0138,  0.0037,
         -0.0129, -0.0259, -0

In [25]:
import torch
import torch.nn.functional as F


def compare_two_token_pt_same_head(
    pt_path_a: str,
    pt_path_b: str,
    *,
    split: str = "tune",
    datapoint_idx: int = 0,
    head_idx: int = 0,
    query_row: int = 0,
):
    """
    Compare pre-head token embeddings from two *_forward_tokens.pt dumps.

    Uses `query_prehead_tokens_by_head` with shape [n_query, num_heads, head_dim].
    Same step_index, same head, full head_dim vector.

    Distance metrics: L1, L2, cosine distance (1 - sim), max|diff|, mean|diff|.
    """
    a = torch.load(pt_path_a, map_location="cpu", weights_only=False)
    b = torch.load(pt_path_b, map_location="cpu", weights_only=False)

    recs_a = a["tokens"][split]
    recs_b = b["tokens"][split]
    idx_a = {int(r["step_index"]): r for r in recs_a}
    idx_b = {int(r["step_index"]): r for r in recs_b}

    if datapoint_idx not in idx_a:
        raise ValueError(f"datapoint_idx={datapoint_idx} not in file A")
    if datapoint_idx not in idx_b:
        raise ValueError(f"datapoint_idx={datapoint_idx} not in file B")

    by_h_a = idx_a[datapoint_idx]["query_prehead_tokens_by_head"]
    by_h_b = idx_b[datapoint_idx]["query_prehead_tokens_by_head"]
    if not isinstance(by_h_a, torch.Tensor):
        by_h_a = torch.as_tensor(by_h_a)
    if not isinstance(by_h_b, torch.Tensor):
        by_h_b = torch.as_tensor(by_h_b)

    nq_a, nh_a, hd_a = by_h_a.shape
    nq_b, nh_b, hd_b = by_h_b.shape
    qr_a = query_row if query_row >= 0 else nq_a + query_row
    qr_b = query_row if query_row >= 0 else nq_b + query_row

    if head_idx < 0 or head_idx >= nh_a or head_idx >= nh_b:
        raise ValueError(f"head_idx out of range: A has {nh_a} heads, B has {nh_b} heads")

    vec_a = by_h_a[qr_a, head_idx, :].float().flatten()
    vec_b = by_h_b[qr_b, head_idx, :].float().flatten()

    if vec_a.shape != vec_b.shape:
        raise ValueError(f"head_dim mismatch: A={vec_a.numel()} vs B={vec_b.numel()}")

    diff = vec_a - vec_b
    l1 = diff.abs().sum().item()
    l2 = torch.norm(diff, p=2).item()
    cos_sim = F.cosine_similarity(vec_a.unsqueeze(0), vec_b.unsqueeze(0), dim=-1).item()
    cos_dist = 1.0 - cos_sim
    max_abs = diff.abs().max().item()
    mean_abs = diff.abs().mean().item()

    print(f"split={split} | step_index={datapoint_idx} | head={head_idx} | query_row={query_row}")
    print(f"head_dim={vec_a.numel()}")
    print(
        f"distances -> L1={l1:.6f} | L2={l2:.6f} | "
        f"cosine_sim={cos_sim:.6f} | cosine_dist={cos_dist:.6f} | "
        f"max|d|={max_abs:.6f} | mean|d|={mean_abs:.6f}"
    )
    print("\nA (full):")
    print(" ".join(f"{v:.6f}" for v in vec_a.tolist()))
    print("\nB (full):")
    print(" ".join(f"{v:.6f}" for v in vec_b.tolist()))
    print("\nA-B (full):")
    print(" ".join(f"{v:+.6f}" for v in diff.tolist()))

    return {
        "vec_a": vec_a,
        "vec_b": vec_b,
        "diff": diff,
        "l1": l1,
        "l2": l2,
        "cosine_similarity": cos_sim,
        "cosine_distance": cos_dist,
        "max_abs_diff": max_abs,
        "mean_abs_diff": mean_abs,
    }


def average_token_distance_all_heads_datapoints(
    pt_path_a: str,
    pt_path_b: str,
    *,
    split: str = "tune",
    query_row: int = 0,
    verbose: bool = True,
) -> dict:
    """
    For every shared datapoint (step_index) and every head, compute L1, L2,
    cosine distance, max|diff|, mean|diff| on the head vector, then average
    across all (datapoint, head) pairs.
    """
    a = torch.load(pt_path_a, map_location="cpu", weights_only=False)
    b = torch.load(pt_path_b, map_location="cpu", weights_only=False)

    recs_a = a["tokens"][split]
    recs_b = b["tokens"][split]
    idx_a = {int(r["step_index"]): r for r in recs_a}
    idx_b = {int(r["step_index"]): r for r in recs_b}
    common = sorted(set(idx_a.keys()) & set(idx_b.keys()))
    if not common:
        raise ValueError("No overlapping step_index between the two token files.")

    l1_list, l2_list, cos_dist_list = [], [], []
    max_abs_list, mean_abs_list = [], []
    nh_a = None

    for step in common:
        by_h_a = idx_a[step]["query_prehead_tokens_by_head"]
        by_h_b = idx_b[step]["query_prehead_tokens_by_head"]
        if not isinstance(by_h_a, torch.Tensor):
            by_h_a = torch.as_tensor(by_h_a)
        if not isinstance(by_h_b, torch.Tensor):
            by_h_b = torch.as_tensor(by_h_b)

        nq_a, nh_a, hd = by_h_a.shape
        nq_b, nh_b, hd_b = by_h_b.shape
        if nh_a != nh_b or hd != hd_b:
            raise ValueError(f"step {step}: shape mismatch A={by_h_a.shape} B={by_h_b.shape}")

        qr_a = query_row if query_row >= 0 else nq_a + query_row
        qr_b = query_row if query_row >= 0 else nq_b + query_row

        va = by_h_a[qr_a].float()
        vb = by_h_b[qr_b].float()
        diff = va - vb

        l1_list.append(diff.abs().sum(dim=-1))
        l2_list.append(torch.norm(diff, p=2, dim=-1))

        va_n = F.normalize(va, dim=-1, eps=1e-8)
        vb_n = F.normalize(vb, dim=-1, eps=1e-8)
        cos_sim = (va_n * vb_n).sum(dim=-1).clamp(-1.0, 1.0)
        cos_dist_list.append(1.0 - cos_sim)

        max_abs_list.append(diff.abs().max(dim=-1).values)
        mean_abs_list.append(diff.abs().mean(dim=-1))

    l1_cat = torch.cat(l1_list)
    l2_cat = torch.cat(l2_list)
    cd_cat = torch.cat(cos_dist_list)
    max_cat = torch.cat(max_abs_list)
    mean_cat = torch.cat(mean_abs_list)

    n_pairs = l1_cat.numel()
    out = {
        "split": split,
        "n_datapoints": len(common),
        "n_heads": int(nh_a) if nh_a is not None else 0,
        "n_pairs": int(n_pairs),
        "mean_l1": float(l1_cat.mean().item()),
        "mean_l2": float(l2_cat.mean().item()),
        "mean_cosine_distance": float(cd_cat.mean().item()),
        "mean_cosine_similarity": float((1.0 - cd_cat).mean().item()),
        "mean_max_abs_diff": float(max_cat.mean().item()),
        "mean_mean_abs_diff": float(mean_cat.mean().item()),
    }

    if verbose:
        print(f"split={split} | datapoints={len(common)} | heads={out['n_heads']} | pairs={n_pairs}")
        for k in (
            "mean_l1",
            "mean_l2",
            "mean_cosine_distance",
            "mean_cosine_similarity",
            "mean_max_abs_diff",
            "mean_mean_abs_diff",
        ):
            print(f"  {k}: {out[k]:.6f}")

    return out


def token_head_norms_single_pt(
    pt_path: str,
    *,
    split: str = "tune",
    query_row: int = 0,
    verbose: bool = True,
) -> dict:
    """
    For one *_forward_tokens.pt file, compute L1 and L2 norms of each head vector
    for each datapoint, then report averages.

    Uses `query_prehead_tokens_by_head` with shape [n_query, num_heads, head_dim].

    L1: sum_i |x_i|  (same as torch.norm(..., p=1, dim=-1))
    L2: sqrt(sum_i x_i^2)  (Euclidean length)

    Returns dict with per-step tensors and means for both norms.
    """
    payload = torch.load(pt_path, map_location="cpu", weights_only=False)
    recs = payload["tokens"][split]

    per_step_head_norms_l1: dict[int, torch.Tensor] = {}
    per_step_head_norms_l2: dict[int, torch.Tensor] = {}
    all_l1: list[torch.Tensor] = []
    all_l2: list[torch.Tensor] = []

    for rec in recs:
        step = int(rec["step_index"])
        by_h = rec["query_prehead_tokens_by_head"]
        if not isinstance(by_h, torch.Tensor):
            by_h = torch.as_tensor(by_h)

        n_query, n_heads, _ = by_h.shape
        qr = query_row if query_row >= 0 else n_query + query_row
        if qr < 0 or qr >= n_query:
            raise ValueError(f"query_row={query_row} out of range at step {step}, n_query={n_query}")

        x = by_h[qr].float()
        # shape [num_heads] each
        norms_l1 = torch.norm(x, p=1, dim=-1)
        norms_l2 = torch.norm(x, p=2, dim=-1)
        per_step_head_norms_l1[step] = norms_l1
        per_step_head_norms_l2[step] = norms_l2
        all_l1.append(norms_l1)
        all_l2.append(norms_l2)

    if not all_l1:
        raise ValueError(f"No token records found for split={split}")

    all_l1_tensor = torch.stack(all_l1, dim=0)  # [n_datapoints, n_heads]
    all_l2_tensor = torch.stack(all_l2, dim=0)

    mean_all_l1 = float(all_l1_tensor.mean().item())
    mean_all_l2 = float(all_l2_tensor.mean().item())
    mean_per_head_l1 = all_l1_tensor.mean(dim=0)
    mean_per_head_l2 = all_l2_tensor.mean(dim=0)

    mean_per_datapoint_l1 = {
        step: float(per_step_head_norms_l1[step].mean().item()) for step in per_step_head_norms_l1
    }
    mean_per_datapoint_l2 = {
        step: float(per_step_head_norms_l2[step].mean().item()) for step in per_step_head_norms_l2
    }

    if verbose:
        print(
            f"split={split} | datapoints={all_l1_tensor.shape[0]} | heads={all_l1_tensor.shape[1]} | "
            f"L1 + L2 norms per head vector"
        )
        print(f"mean_over_all_datapoints_and_heads (L1): {mean_all_l1:.6f}")
        print(f"mean_over_all_datapoints_and_heads (L2): {mean_all_l2:.6f}")
        print("mean_per_head (L1):", [round(float(v), 6) for v in mean_per_head_l1.tolist()])
        print("mean_per_head (L2):", [round(float(v), 6) for v in mean_per_head_l2.tolist()])

    return {
        "split": split,
        "n_datapoints": int(all_l1_tensor.shape[0]),
        "n_heads": int(all_l1_tensor.shape[1]),
        "per_step_head_norms_l1": per_step_head_norms_l1,
        "per_step_head_norms_l2": per_step_head_norms_l2,
        "mean_over_all_datapoints_and_heads_l1": mean_all_l1,
        "mean_over_all_datapoints_and_heads_l2": mean_all_l2,
        "mean_per_head_l1": mean_per_head_l1,
        "mean_per_head_l2": mean_per_head_l2,
        "mean_per_datapoint_l1": mean_per_datapoint_l1,
        "mean_per_datapoint_l2": mean_per_datapoint_l2,
        # Back-compat aliases (L2 was the previous default)
        "per_step_head_norms": per_step_head_norms_l2,
        "mean_over_all_datapoints_and_heads": mean_all_l2,
        "mean_per_head": mean_per_head_l2,
        "mean_per_datapoint": mean_per_datapoint_l2,
    }



In [26]:
compare_two_token_pt_same_head(
    "outputs/attention_dumps/bitcoin_synthetic_additive_toy_forward_tokens.pt",
    "outputs/attention_dumps/bitcoin_synthetic_additive_forward_tokens.pt",
    split="tune",       # or "eval"
    datapoint_idx=0,    # same as step_index in the dump
    head_idx=0,         # 0..7 for 8 heads
    query_row=0,        # single query per step → use 0
)

split=tune | step_index=0 | head=0 | query_row=0
head_dim=96
distances -> L1=106.221466 | L2=13.539991 | cosine_sim=0.956477 | cosine_dist=0.043523 | max|d|=3.715207 | mean|d|=1.106474

A (full):
4.646335 -0.150109 -1.067197 -6.425202 1.375556 -8.085239 -4.566921 -8.252032 1.566780 -1.286464 -0.186870 -0.805907 -6.359932 0.989197 -0.476522 1.445272 -1.657675 -10.201097 -0.645724 -5.085265 2.227809 -1.187147 -1.619421 4.048172 10.344462 0.510385 2.876309 3.923501 2.500182 14.693568 5.851725 2.094628 2.359153 3.900948 -1.235890 0.708115 -2.699826 -3.125350 -3.255738 0.252285 4.133563 -6.892526 0.815183 1.027038 4.386126 -2.668753 -3.234158 1.671896 1.307018 -1.272065 -3.461259 1.822334 -0.424443 -0.543127 -7.119343 0.081785 -3.590038 0.164874 2.158683 1.783899 -2.372168 -4.446042 -2.717315 1.988288 0.383818 1.548277 -0.734723 -2.608021 -3.682477 3.243654 3.340720 -1.205979 -4.586868 -4.219176 -12.864104 -7.283213 -5.198591 1.039011 2.136683 2.313502 -1.668851 -0.986726 2.004071 5.705389 

{'vec_a': tensor([ 4.6463e+00, -1.5011e-01, -1.0672e+00, -6.4252e+00,  1.3756e+00,
         -8.0852e+00, -4.5669e+00, -8.2520e+00,  1.5668e+00, -1.2865e+00,
         -1.8687e-01, -8.0591e-01, -6.3599e+00,  9.8920e-01, -4.7652e-01,
          1.4453e+00, -1.6577e+00, -1.0201e+01, -6.4572e-01, -5.0853e+00,
          2.2278e+00, -1.1871e+00, -1.6194e+00,  4.0482e+00,  1.0344e+01,
          5.1039e-01,  2.8763e+00,  3.9235e+00,  2.5002e+00,  1.4694e+01,
          5.8517e+00,  2.0946e+00,  2.3592e+00,  3.9009e+00, -1.2359e+00,
          7.0811e-01, -2.6998e+00, -3.1254e+00, -3.2557e+00,  2.5229e-01,
          4.1336e+00, -6.8925e+00,  8.1518e-01,  1.0270e+00,  4.3861e+00,
         -2.6688e+00, -3.2342e+00,  1.6719e+00,  1.3070e+00, -1.2721e+00,
         -3.4613e+00,  1.8223e+00, -4.2444e-01, -5.4313e-01, -7.1193e+00,
          8.1785e-02, -3.5900e+00,  1.6487e-01,  2.1587e+00,  1.7839e+00,
         -2.3722e+00, -4.4460e+00, -2.7173e+00,  1.9883e+00,  3.8382e-01,
          1.5483e+00, -7.3472

In [27]:
average_token_distance_all_heads_datapoints(
    "outputs/attention_dumps/bitcoin_synthetic_additive_toy_forward_tokens.pt",
    "outputs/attention_dumps/bitcoin_synthetic_additive_forward_tokens.pt",
    split="tune",
    query_row=0,
)

split=tune | datapoints=10 | heads=8 | pairs=80
  mean_l1: 139.903168
  mean_l2: 17.925446
  mean_cosine_distance: 0.101153
  mean_cosine_similarity: 0.898847
  mean_max_abs_diff: 5.381663
  mean_mean_abs_diff: 1.457325


{'split': 'tune',
 'n_datapoints': 10,
 'n_heads': 8,
 'n_pairs': 80,
 'mean_l1': 139.90316772460938,
 'mean_l2': 17.925445556640625,
 'mean_cosine_distance': 0.10115285962820053,
 'mean_cosine_similarity': 0.898847222328186,
 'mean_max_abs_diff': 5.381662845611572,
 'mean_mean_abs_diff': 1.4573246240615845}

In [30]:
res = token_head_norms_single_pt(
    "outputs/attention_dumps/bitcoin_synthetic_additive_toy_forward_tokens.pt",
    split="tune",
    query_row=0,
)

split=tune | datapoints=10 | heads=8 | norm=L2
mean_over_all_datapoints_and_heads: 40.556156
mean_per_head: [41.453331, 37.324635, 38.609898, 46.841591, 40.021687, 40.307564, 41.363369, 38.527157]


In [31]:
res = token_head_norms_single_pt(
    "outputs/attention_dumps/bitcoin_synthetic_additive_forward_tokens.pt",
    split="tune",
    query_row=0,
)

split=tune | datapoints=10 | heads=8 | norm=L2
mean_over_all_datapoints_and_heads: 38.492737
mean_per_head: [41.012959, 34.666851, 34.537762, 43.24519, 38.44014, 38.967773, 39.313671, 37.757549]
